Basic Components
====

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import tiktoken

### 1. Download data

In [2]:

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


url = "https://www.gutenberg.org/ebooks/2701.txt.utf-8"
repo_root = find_repo_root()
data_dir = repo_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

text_path = data_dir / "moby_dick.txt"

if not text_path.exists():
    urlretrieve(url, text_path)

text = text_path.read_text(encoding="utf-8")

print(f"Saved to: {text_path}")
print(f"Characters: {len(text):,}")
print(text[:500])


Saved to: /home/pyml/git_vmirly/llm-recipes/data/raw/moby_dick.txt
Characters: 1,238,279
The Project Gutenberg eBook of Moby Dick; Or, The Whale
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBo


### 2. Tokenization

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")
token_ids = tokenizer.encode(text)

n_vocab = tokenizer.n_vocab

# number of unique tokens in the tokenizer's vocabulary
print(f"Number of unique tokens: {n_vocab}")

print(f"Tokens: {len(token_ids):,}")
print(f"First 10 token IDs: {token_ids[:10]}")

# decoding:
decoded_text = tokenizer.decode(token_ids[:10])
print(f"Decoded text: {decoded_text}")


Number of unique tokens: 50257
Tokens: 331,691
First 10 token IDs: [464, 4935, 20336, 46566, 286, 16540, 88, 11740, 26, 1471]
Decoded text: The Project Gutenberg eBook of Moby Dick; Or


### 3. Embeddings

In [4]:
embedding_dim = 768
context_length = 1024

In [5]:
import torch
import torch.nn as nn

token_embeddings = nn.Embedding(
    num_embeddings=n_vocab, embedding_dim=embedding_dim
)

print(f"Token embeddings shape: {token_embeddings.weight.shape}")

# testing:
input_ids = torch.tensor(token_ids[:10])
tok_emb = token_embeddings(input_ids)

print(f"input_ids.shape: {input_ids.shape}")   # [10]
print(f"tok_emb.shape: {tok_emb.shape}")     # [10, 768]

tok_emb

Token embeddings shape: torch.Size([50257, 768])
input_ids.shape: torch.Size([10])
tok_emb.shape: torch.Size([10, 768])


tensor([[ 0.1528,  0.2416,  0.9716,  ...,  0.5706,  0.2511, -0.0896],
        [ 1.1758, -0.5099, -1.8061,  ...,  0.8756, -2.0066,  0.0266],
        [-0.0703, -1.2523,  1.9039,  ...,  0.5348, -0.1129,  0.7674],
        ...,
        [ 0.3308, -1.6435, -0.0323,  ..., -0.6222, -0.6475, -0.3098],
        [-0.0731,  0.7867,  1.3753,  ..., -0.9731,  0.5617, -1.3173],
        [ 0.6622, -0.0721,  0.1312,  ..., -1.3847,  0.8328,  0.5427]],
       grad_fn=<EmbeddingBackward0>)

In [6]:
pos_embeddings = nn.Embedding(
    num_embeddings=context_length, embedding_dim=embedding_dim
)

seq_len = input_ids.shape[0]

positions = torch.arange(seq_len)
pos_emb = pos_embeddings(positions)
print(f"pos_emb.shape: {pos_emb.shape}")     # [10, 768]

pos_emb

pos_emb.shape: torch.Size([10, 768])


tensor([[-1.4411, -0.6638, -0.8416,  ...,  1.0388,  1.4541,  0.4105],
        [ 0.8597, -0.9896,  0.0060,  ...,  1.1319, -0.2347, -0.4796],
        [-0.0951,  0.0284,  0.0327,  ...,  1.0042, -0.3161, -1.4430],
        ...,
        [-1.0484, -0.6317,  1.3937,  ...,  1.0080,  0.8659, -0.4836],
        [ 0.0870, -0.1970, -0.3925,  ..., -0.2989,  0.6485,  0.7339],
        [-1.4835,  1.3259, -1.1810,  ...,  0.9171,  0.3266, -0.2469]],
       grad_fn=<EmbeddingBackward0>)

In [7]:
# adding token embeddings and positional embeddings

x = tok_emb + pos_emb
print(f"x.shape: {x.shape}")  # [10, 768]

x.shape: torch.Size([10, 768])


### 4. Self-attention

Notes:
  - This is bidirectional attention, because every token can attend to every other token.
  - (GPT needs causal self-attention, where token i can only attend to tokens <= i)

In [8]:
# bidirectional attention

class SelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, v)

        return output, attn_weights


# testing the self-attention layer
attention_layer = SelfAttention(embedding_dim=embedding_dim)
output, attn_weights = attention_layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

print(f"Attention weights shape: {attn_weights.shape}")
print(
    "Sum of attention weights for each token (should be 1):\n",
    attn_weights.sum(dim=-1)
)

Input shape: torch.Size([10, 768])
Output shape: torch.Size([10, 768])
Attention weights shape: torch.Size([10, 10])
Sum of attention weights for each token (should be 1):
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)


In [9]:
# causal self-attention

class CausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        seq_len = x.shape[-2]  # this works for both single and batched inputs
        attn_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1,
        )

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
        # apply the mask
        attn_scores = attn_scores.masked_fill(attn_mask, float("-inf"))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, v)

        return output, attn_weights
    
# testing
causal_attention_layer = CausalSelfAttention(embedding_dim=embedding_dim)
output_causal, attn_weights = causal_attention_layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output_causal.shape}")

print(
    "Sum of attention weights for each token (should be 1):\n",
    attn_weights.sum(dim=-1)
)

# verify that the attention weights are zero for future tokens
print(
    "Attention weights must be zero for future tokens:\n",
    attn_weights
)

Input shape: torch.Size([10, 768])
Output shape: torch.Size([10, 768])
Sum of attention weights for each token (should be 1):
 tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000], grad_fn=<SumBackward1>)
Attention weights must be zero for future tokens:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.2620, 0.7380, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.4292, 0.1414, 0.4293, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1441, 0.1594, 0.5839, 0.1126, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0321, 0.3482, 0.1784, 0.2177, 0.2237, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1750, 0.1226, 0.2910, 0.1126, 0.0714, 0.2274, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1515, 0.0441, 0.1639, 0.0630, 0.4597, 0.0602, 0.0577, 0.0000, 0.0000,
         0.0000],
        

### 5. Multi-head self-attention (MHA)

In [10]:
# v1: naive implementation using for-loops
#
# This is only for illustration.
# Each head receives the full embedding_dim input and also outputs embedding_dim.
# Therefore, concatenating num_heads heads produces an output of size:
#     embedding_dim * num_heads
#
# This is NOT the standard Transformer MHA implementation.
# In standard MHA, each head projects the full input down to head_dim:
#     head_dim = embedding_dim // num_heads
# and concatenating all heads gives back embedding_dim.
   
class MultiheadSelfAttention_v1(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalSelfAttention(embedding_dim) for _ in range(num_heads)
        ])

    def forward(self, x):
        head_outputs = []
        for head in self.heads:
            head_output, _ = head(x)
            head_outputs.append(head_output)

        # concatenate the outputs
        output = torch.cat(head_outputs, dim=-1)
        return output

# testing
mha_layer_v1 = MultiheadSelfAttention_v1(embedding_dim=embedding_dim, num_heads=8)

output_mha_v1 = mha_layer_v1(x)
print(f"Input shape: {x.shape}")
print(f"MHA (v1) Output shape: {output_mha_v1.shape}")

Input shape: torch.Size([10, 768])
MHA (v1) Output shape: torch.Size([10, 6144])


In [11]:
# v2: standard implementation of multi-head attention

class MultiheadSelfAttention_v2(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        self.w_q = nn.Linear(embedding_dim, embedding_dim)
        self.w_k = nn.Linear(embedding_dim, embedding_dim)
        self.w_v = nn.Linear(embedding_dim, embedding_dim)
        self.w_o = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()

        # Linear projections
        q = self.w_q(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.w_k(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_v(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # Causal mask
        attn_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        attn_scores = attn_scores.masked_fill(attn_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_weights, v)

        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        output = self.w_o(output)

        return output
    
# testing:
mha_layer_v2 = MultiheadSelfAttention_v2(embedding_dim=embedding_dim, num_heads=8)
output_mha_v2 = mha_layer_v2(x.unsqueeze(0))  # add batch dimension
print(f"Input shape: {x.unsqueeze(0).shape}")
print(f"MHA (v2) Output shape: {output_mha_v2.shape}")

Input shape: torch.Size([1, 10, 768])
MHA (v2) Output shape: torch.Size([1, 10, 768])


## References

 * OpenAI GPT-2 model (TensorFlow): https://github.com/openai/gpt-2/blob/master/src/model.py